# META-CXR Stage 1 — private 2×T4 DDP session

One notebook session trains a configurable number of complete epochs, defaults to one, and reserves 90 minutes for private checkpoint publication. The held-out test stays disabled here by default; run the sensitivity notebook once after the final validation-selected checkpoint.

In [ ]:
DATASET_SLUG = "phuong20052/mimic-cxr-jpg-dataset"   # attached private dataset slug
CHECKPOINT_INPUT_SLUG = ""            # prior private checkpoint dataset slug; blank for session 1
CHECKPOINT_DATASET_HANDLE = "phuong20052/meta-cxr-checkpoints"   # pre-created private owner/slug for upload
REPO_COMMIT = "b3e10f480febda49d0d2ad6e01d6ae2ec86a241e"   # exact 40-character smoke-repo commit
SESSION_INDEX = 1
SESSION_EPOCHS = 1
TOTAL_PLANNED_EPOCHS = 10                 # fixed LR-schedule horizon across every resume session
SEED = 42
BATCH_PER_GPU = 1
ACCUMULATION = 64
NUM_WORKERS = 4
SESSION_HOURS = 12.0
UPLOAD_RESERVE_MINUTES = 90.0
FINALIZE_TEST = False                     # keep False when notebook 02 will perform final test sensitivity


In [ ]:
import os, pathlib, subprocess, sys
if not DATASET_SLUG or not CHECKPOINT_DATASET_HANDLE:
    raise ValueError("DATASET_SLUG and CHECKPOINT_DATASET_HANDLE are required")
if len(REPO_COMMIT) != 40 or any(c not in '0123456789abcdef' for c in REPO_COMMIT.lower()):
    raise ValueError("REPO_COMMIT must be an exact 40-character SHA")
repo_dir = pathlib.Path('/kaggle/working/META-CXR-SMOKETEST')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/minhphuong150505/META-CXR-SMOKETEST.git', str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'fetch', '--depth=1', 'origin', REPO_COMMIT], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', REPO_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
if actual != REPO_COMMIT:
    raise RuntimeError('Exact commit checkout failed')
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))


In [ ]:
# PyTorch 2.6 changed torch.load's default. Resume checkpoints intentionally contain optimizer and per-rank RNG state.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'


In [ ]:
# Load required Kaggle secrets into this process without printing values.
from smoke.runtime import load_kaggle_secrets
load_kaggle_secrets(
    ('GCS_SERVICE_ACCOUNT', 'WANDB_API_KEY', 'HF_TOKEN', 'KAGGLE_API_TOKEN'),
    '/kaggle/working/.meta-cxr-secrets',
)
print('Loaded required secrets into OS environment (values hidden).')


In [ ]:
import json
from smoke.runtime import environment_fingerprint, assert_two_t4
before = environment_fingerprint()
print(json.dumps(before, indent=2, sort_keys=True))
assert_two_t4(before)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', 'requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', 'hi-ml-multimodal==0.2.1'], check=True)
after = environment_fingerprint()
assert_two_t4(after)
print(json.dumps(after, indent=2, sort_keys=True))


In [ ]:
from smoke.runtime import compatibility_matrix
compatibility = compatibility_matrix(before, after)
print(json.dumps(compatibility, indent=2, sort_keys=True))


In [ ]:
from smoke.runtime import discover_dataset, load_dataset_manifest, write_runtime_env_config
dataset_root = discover_dataset(DATASET_SLUG)
dataset_manifest, manifest_path, dataset_hash = load_dataset_manifest(dataset_root)
if dataset_manifest.get('status') != 'qa_passed':
    raise RuntimeError('Dataset manifest is not QA-passed')
write_runtime_env_config(dataset_root, '/kaggle/working/meta-cxr-output')
subprocess.run([sys.executable, 'scripts/kaggle_data_preflight.py', '--dataset-root', str(dataset_root), '--seed', str(SEED)], check=True)
print({'dataset_manifest_sha256': dataset_hash, 'actual_bytes': dataset_manifest.get('actual_bytes'), 'counts': dataset_manifest.get('counts')})


In [ ]:
import hashlib, json, math, shutil, torch
if TOTAL_PLANNED_EPOCHS < SESSION_EPOCHS:
    raise ValueError('TOTAL_PLANNED_EPOCHS must cover the requested session')
identity_payload = {
    'source_commit': REPO_COMMIT, 'dataset_manifest_sha256': dataset_hash,
    'encoders': ['biovil', 'pubmedclip', 'swin'], 'multi_view': True,
    'image_size': 448, 'batch_per_gpu': BATCH_PER_GPU, 'world_size': 2,
    'accumulation': ACCUMULATION, 'seed': SEED, 'scheduler_max_epoch': TOTAL_PLANNED_EPOCHS,
    'selection_metric': 'f1_positive_macro_defined_only',
    'uncertain_policy': 'ignore_uncertain',
}
config_fingerprint = hashlib.sha256(json.dumps(identity_payload, sort_keys=True).encode()).hexdigest()
run_name = f"meta-cxr-e123-{dataset_hash[:8]}-seed{SEED}-session{SESSION_INDEX:02d}"
output_base = pathlib.Path('/kaggle/working/meta-cxr-output')
run_dir = output_base / run_name
resume_path = None
start_epoch = 0
if CHECKPOINT_INPUT_SLUG:
    prior_root = pathlib.Path('/kaggle/input') / CHECKPOINT_INPUT_SLUG.split('/')[-1]
    prior_last = prior_root / 'checkpoint_last.pth'
    prior_best = prior_root / 'checkpoint_best.pth'
    if not prior_last.is_file() or not prior_best.is_file():
        raise FileNotFoundError('Prior checkpoint dataset is missing best/last files')
    prior = torch.load(prior_last, map_location='cpu')
    expected = {'source_commit': REPO_COMMIT, 'dataset_manifest_sha256': dataset_hash, 'config_fingerprint': config_fingerprint}
    if prior.get('identity') != expected:
        raise RuntimeError('Prior checkpoint identity does not match this run')
    start_epoch = int(prior['epoch']) + 1
    run_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(prior_last, run_dir / 'checkpoint_last.pth')
    shutil.copy2(prior_best, run_dir / 'checkpoint_best.pth')
    resume_path = run_dir / 'checkpoint_last.pth'
max_epoch = start_epoch + SESSION_EPOCHS
if max_epoch > TOTAL_PLANNED_EPOCHS:
    raise ValueError('Session would exceed TOTAL_PLANNED_EPOCHS')
train_studies = int(dataset_manifest['counts']['split_studies']['train'])
microbatches_per_rank = math.ceil(train_studies / 2 / BATCH_PER_GPU)
optimizer_steps = math.ceil(microbatches_per_rank / ACCUMULATION)
warmup_steps = max(1, math.ceil(optimizer_steps * 0.10))
print({'run_name': run_name, 'start_epoch': start_epoch, 'max_epoch': max_epoch, 'scheduler_max_epoch': TOTAL_PLANNED_EPOCHS, 'optimizer_steps_per_epoch': optimizer_steps, 'effective_batch': BATCH_PER_GPU * 2 * ACCUMULATION, 'warmup_steps': warmup_steps})


In [ ]:
# Two-rank construction + forward/backward/all-reduce/checkpoint preflight (2 optimizer steps).
preflight_name = f'preflight-{dataset_hash[:8]}-seed{SEED}'
preflight_dir = pathlib.Path('/kaggle/working/meta-cxr-preflight') / preflight_name
preflight_fingerprint = hashlib.sha256((config_fingerprint + ':preflight').encode()).hexdigest()
preflight_cmd = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', '-m', 'pretraining.train', '--cfg-path', 'pretraining/configs/stage1_smoke_2xt4.yaml', '--options',
    f'run.run_name={preflight_name}', 'run.output_dir=/kaggle/working/meta-cxr-preflight', f'run.source_commit={REPO_COMMIT}', f'run.dataset_manifest_sha256={dataset_hash}', f'run.config_fingerprint={preflight_fingerprint}',
    f'run.batch_size_train={BATCH_PER_GPU}', f'run.accum_grad_iters={ACCUMULATION}', f'run.num_workers={NUM_WORKERS}', 'run.warmup_steps=1', 'run.max_epoch=1', f'run.truncate_train={BATCH_PER_GPU * 2 * ACCUMULATION * 2}', 'run.truncate_val=8', 'run.test_splits=[]', 'run.finalize_test=false', 'run.save_predictions=false']
if not (preflight_dir / 'checkpoint_last.pth').is_file():
    subprocess.run(preflight_cmd, check=True)
log_lines = (preflight_dir / 'log.txt').read_text().splitlines()
train_logs = [json.loads(line) for line in log_lines if line.startswith('{"train_') and 'train_epoch_wall_seconds' in line]
if not train_logs:
    raise RuntimeError('DDP preflight did not emit timing evidence')
probe = train_logs[-1]
if not math.isfinite(float(probe['train_loss'])):
    raise RuntimeError('DDP preflight loss is not finite')
print(probe)


In [ ]:
from smoke.runtime import assert_session_eta
eta = assert_session_eta(int(probe['train_optimizer_steps']), float(probe['train_epoch_wall_seconds']), optimizer_steps, SESSION_HOURS, UPLOAD_RESERVE_MINUTES)
print(eta)


In [ ]:
full_cmd = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', '-m', 'pretraining.train', '--cfg-path', 'pretraining/configs/stage1_smoke_2xt4.yaml', '--options',
    f'run.run_name={run_name}', f'run.source_commit={REPO_COMMIT}', f'run.dataset_manifest_sha256={dataset_hash}', f'run.config_fingerprint={config_fingerprint}',
    f'run.batch_size_train={BATCH_PER_GPU}', f'run.accum_grad_iters={ACCUMULATION}', f'run.num_workers={NUM_WORKERS}', f'run.warmup_steps={warmup_steps}', f'run.max_epoch={max_epoch}', f'run.scheduler_max_epoch={TOTAL_PLANNED_EPOCHS}', f'run.seed={SEED}', f'run.finalize_test={str(FINALIZE_TEST).lower()}']
if resume_path is not None:
    full_cmd.append(f'run.resume_ckpt_path={resume_path}')
completed = False
if (run_dir / 'checkpoint_last.pth').is_file():
    current = torch.load(run_dir / 'checkpoint_last.pth', map_location='cpu')
    completed = int(current.get('epoch', -1)) >= max_epoch - 1 and current.get('identity', {}).get('config_fingerprint') == config_fingerprint
if not completed:
    subprocess.run(full_cmd, check=True)
last = torch.load(run_dir / 'checkpoint_last.pth', map_location='cpu')
if int(last['epoch']) != max_epoch - 1:
    raise RuntimeError('Session did not finish the requested complete epoch count')
print({'completed_epoch': int(last['epoch']), 'run_dir': str(run_dir), 'peak_vram_each_rank': 'recorded in Kaggle nvidia-smi/runtime logs'})


In [ ]:
run_manifest = {
    'source_commit': REPO_COMMIT,
    'dataset_manifest_sha256': dataset_hash,
    'config_fingerprint': config_fingerprint,
    'session_index': SESSION_INDEX,
    'completed_epoch': int(last['epoch']),
    'training': {
        'encoders': ['biovil', 'pubmedclip', 'swin'], 'multi_view': True,
        'image_size': 448, 'batch_per_gpu': BATCH_PER_GPU, 'world_size': 2,
        'accumulation': ACCUMULATION, 'effective_batch': BATCH_PER_GPU * 2 * ACCUMULATION,
        'optimizer_steps_per_epoch': optimizer_steps, 'scheduler_max_epoch': TOTAL_PLANNED_EPOCHS,
        'warmup_steps': warmup_steps, 'amp': 'fp16', 'seed': SEED, 'num_workers': NUM_WORKERS,
        'selection_metric': 'f1_positive_macro_defined_only', 'uncertain_policy': 'ignore_uncertain',
    },
    'eta_probe': eta,
    'environment': after,
    'held_out_test_finalized': FINALIZE_TEST,
}
(run_dir / 'run_manifest.json').write_text(json.dumps(run_manifest, indent=2, sort_keys=True))


In [ ]:
from smoke.checkpoints import write_artifact_manifest, upload_private_checkpoint_dataset
identity = {'source_commit': REPO_COMMIT, 'dataset_manifest_sha256': dataset_hash, 'config_fingerprint': config_fingerprint}
write_artifact_manifest(run_dir, identity)
upload_private_checkpoint_dataset(CHECKPOINT_DATASET_HANDLE, run_dir)
print('Private checkpoint dataset upload and manifest verification succeeded; local files retained.')
